# 02 — Data Completeness & Phase 0.2 Source Integration

**Purpose:** Validate the Phase 0.2 ingestion layer. Goals:
1. Build the merged 2024-25 player dataset (nba_api + BBRef + salary + EPM)
2. Show coverage: what percentage of players have data from each source?
3. Inspect the Lakers roster with all merged columns
4. Show Luka's full merged row — every metric we now have
5. Top 20 by BPM and VORP (now available from BBRef)
6. Surface any join failures or data quality issues
7. Lakers-specific shooting + defense breakdown (catch-and-shoot 3PT next to defensive FG% suppression)

**New sources added in Phase 0.2:**
- `src/data/bbref_stats.py` — PER, TS%, WS, WS/48, BPM, DBPM, OBPM, VORP
- `src/data/salaries.py` — salary data (HoopsHype scraper + manual CSV loader)
- `src/data/advanced_metrics.py` — EPM (Dunks & Threes; scraper + manual loader)
- `src/data/merge.py` — unified build_player_dataset() + data_completeness_report()

**Do not re-run all cells repeatedly** — BBRef scraping respects rate limits (3.5s delay)
and the first run caches everything to `data/cache/`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/suveerdhawan/Desktop/Codex/lakers-trade-engine


In [2]:
import logging

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(name)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)

from src.data import cache as C
from src.data import bbref_stats as B
from src.data import advanced_metrics as AM
from src.data import nba_stats as N
from src.data.merge import build_player_dataset, completeness_summary, data_completeness_report
from src.data.config import CACHE_DIR, CURRENT_SEASON

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

SEASON = "2024-25"
LUKA_ID = 1629029

print(f"Cache dir : {CACHE_DIR}")
print(f"Target season: {SEASON}")

Cache dir : /Users/suveerdhawan/Desktop/Codex/lakers-trade-engine/data/cache
Target season: 2024-25


---
## 1. Build the Merged Dataset

`build_player_dataset()` runs the full merge pipeline:
- nba_api base + advanced + shooting + defense (PLAYER_ID joins)
- BBRef advanced (normalized name join)
- HoopsHype salary scrape (normalized name join)
- EPM from Dunks & Threes (normalized name join, may be empty if scraping blocked)

First run will hit BBRef and HoopsHype. Subsequent runs are cache-only.

In [3]:
df = build_player_dataset(SEASON)
print(f"Dataset shape: {df.shape}")
print(f"Columns ({len(df.columns)}):")
for i, col in enumerate(df.columns):
    print(f"  {i:3d}  {col}")

15:13:20  src.data.merge  INFO  Building player dataset for 2024-25 ...
15:13:20  src.data.cache  INFO  cache hit: nba_player_stats_2024-25_PerGame
15:13:20  src.data.cache  INFO  cache hit: nba_player_advanced_2024-25
15:13:20  src.data.cache  INFO  cache hit: nba_shooting_catchshoot_2024-25
15:13:20  src.data.cache  INFO  cache hit: nba_shooting_pullup_2024-25
15:13:20  src.data.cache  INFO  cache hit: nba_player_defense_2024-25
15:13:20  src.data.cache  INFO  cache miss -- fetching: bbref_advanced_2025
15:13:25  src.data.bbref_stats  WARNING  read_html failed: `Import lxml` failed.  Use pip or conda to install the lxml package.
15:13:25  src.data.cache  INFO  cached 0 rows to bbref_advanced_2025.parquet
15:13:25  src.data.merge  WARNING  BBRef advanced empty for season 2025 - skipping BBRef join
15:13:25  src.data.cache  INFO  cache miss -- fetching: salaries_hoopshype_2024-25
15:13:30  src.data.salaries  WARNING  Failed to parse HoopsHype table: `Import lxml` failed.  Use pip or co

Dataset shape: (569, 148)
Columns (148):
    0  PLAYER_ID
    1  PLAYER_NAME
    2  NICKNAME
    3  TEAM_ID
    4  TEAM_ABBREVIATION
    5  AGE
    6  GP
    7  W
    8  L
    9  W_PCT
   10  MIN
   11  FGM
   12  FGA
   13  FG_PCT
   14  FG3M
   15  FG3A
   16  FG3_PCT
   17  FTM
   18  FTA
   19  FT_PCT
   20  OREB
   21  DREB
   22  REB
   23  AST
   24  TOV
   25  STL
   26  BLK
   27  BLKA
   28  PF
   29  PFD
   30  PTS
   31  PLUS_MINUS
   32  NBA_FANTASY_PTS
   33  DD2
   34  TD3
   35  WNBA_FANTASY_PTS
   36  GP_RANK
   37  W_RANK
   38  L_RANK
   39  W_PCT_RANK
   40  MIN_RANK
   41  FGM_RANK
   42  FGA_RANK
   43  FG_PCT_RANK
   44  FG3M_RANK
   45  FG3A_RANK
   46  FG3_PCT_RANK
   47  FTM_RANK
   48  FTA_RANK
   49  FT_PCT_RANK
   50  OREB_RANK
   51  DREB_RANK
   52  REB_RANK
   53  AST_RANK
   54  TOV_RANK
   55  STL_RANK
   56  BLK_RANK
   57  BLKA_RANK
   58  PF_RANK
   59  PFD_RANK
   60  PTS_RANK
   61  PLUS_MINUS_RANK
   62  NBA_FANTASY_PTS_RANK
   63  DD2_RANK
   64

---
## 2. Data Completeness Report

How many players have data from each source? This exposes join failures and source gaps before we rely on any metric in feature engineering.

In [4]:
summary = completeness_summary(df)
print("=== Data Completeness Report ===")
display(summary)

# Flag any source below 80% coverage (excluding EPM which may not be scraped)
core_sources = summary[~summary["source"].str.contains("epm")]
low_coverage = core_sources[core_sources["coverage_pct"] < 80]
if not low_coverage.empty:
    print("\nWARNING - low coverage sources (< 80%):")
    display(low_coverage)
else:
    print("\nAll core sources at 80%+ coverage.")

=== Data Completeness Report ===


,source,sentinel_col,coverage_pct,n_players,n_total
0,nba_api_base,PTS,100.000,569,569
1,nba_api_advanced,NET_RATING,100.000,569,569
2,nba_api_defense,PCT_PLUSMINUS,99.800,568,569
3,nba_api_shooting,CS_CATCH_SHOOT_FG3_PCT,94.700,539,569
4,bbref_advanced,BPM,0.000,0,569
5,bbref_ws,WS,0.000,0,569
6,bbref_per,PER,0.000,0,569
7,bbref_vorp,VORP,0.000,0,569
8,salary,SALARY,0.000,0,569
9,epm,EPM,0.000,0,569



WARNING - low coverage sources (< 80%):


,source,sentinel_col,coverage_pct,n_players,n_total
4,bbref_advanced,BPM,0.000,0,569
5,bbref_ws,WS,0.000,0,569
6,bbref_per,PER,0.000,0,569
7,bbref_vorp,VORP,0.000,0,569
8,salary,SALARY,0.000,0,569


In [5]:
# Show cache state after building the dataset
cache_df = C.cache_info()
print(f"Total cached files: {len(cache_df)}")
display(cache_df)

Total cached files: 22


,file,key,size_kb,modified
0,bbref_advanced_2025.parquet,bbref_advanced_2025,0.600,2026-05-28 15:13
1,epm_2024-25.parquet,epm_2024-25,3.800,2026-05-28 15:13
2,nba_player_advanced_2019-20.parquet,nba_player_advanced_2019-20,201.500,2026-05-27 23:48
3,nba_player_advanced_2020-21.parquet,nba_player_advanced_2020-21,204.500,2026-05-27 23:48
4,nba_player_advanced_2021-22.parquet,nba_player_advanced_2021-22,219.200,2026-05-27 23:48
5,nba_player_advanced_2022-23.parquet,nba_player_advanced_2022-23,203.700,2026-05-27 23:48
6,nba_player_advanced_2023-24.parquet,nba_player_advanced_2023-24,212.400,2026-05-27 23:48
7,nba_player_advanced_2024-25.parquet,nba_player_advanced_2024-25,212.900,2026-05-27 23:48
8,nba_player_advanced_2025-26.parquet,nba_player_advanced_2025-26,216.100,2026-05-27 23:48
9,nba_player_advanced_2026-27.parquet,nba_player_advanced_2026-27,33.500,2026-05-27 23:48


---
## 3. Lakers Roster — Full Merged View

The roster with all available columns: nba_api metrics, BBRef advanced (PER, WS, BPM, VORP), and salary. This is the first time we see all these metrics in one row per player.

In [6]:
lakers = (
    df[df["TEAM_ABBREVIATION"] == "LAL"]
    .sort_values("MIN", ascending=False)
    .reset_index(drop=True)
)
print(f"Lakers roster: {len(lakers)} players")

# Core display: name + key metrics from each source
core_cols = [c for c in [
    "PLAYER_NAME", "AGE", "GP", "MIN",
    # nba_api base
    "PTS", "AST", "REB", "TS_PCT", "USG_PCT", "NET_RATING",
    # BBRef
    "PER", "WS", "WS_48", "BPM", "VORP",
    # Salary
    "SALARY",
    # EPM (if available)
    "EPM",
] if c in lakers.columns]

display(lakers[core_cols])

Lakers roster: 21 players


,PLAYER_NAME,AGE,GP,MIN,PTS,AST,REB,TS_PCT,USG_PCT,NET_RATING
0,Luka Dončić,26.000,50,35.400,28.200,7.700,8.200,0.587,0.328,9.100
1,Austin Reaves,27.000,73,34.900,20.200,5.800,4.500,0.616,0.230,3.800
2,LeBron James,40.000,70,34.900,24.400,8.200,7.800,0.604,0.291,-1.300
3,Rui Hachimura,27.000,59,31.700,13.100,1.400,5.000,0.619,0.156,3.900
4,Dorian Finney-Smith,32.000,63,28.900,8.700,1.400,3.900,0.604,0.123,8.800
5,Gabe Vincent,29.000,72,21.200,6.400,1.400,1.300,0.536,0.131,1.100
6,Jaxson Hayes,25.000,56,19.500,6.800,1.000,4.800,0.720,0.120,2.500
7,Dalton Knecht,24.000,78,19.200,9.100,0.800,2.800,0.594,0.181,-2.800
8,Maxi Kleber,33.000,34,18.700,3.000,1.300,2.800,0.489,0.085,-0.400
9,Jordan Goodwin,26.000,29,18.700,5.600,1.400,3.900,0.541,0.134,-1.200


---
## 4. Luka's Full Merged Row

Every metric we now have for Luka Doncic in 2024-25. This is the benchmark row for the Luka Complement Score — we want to understand his profile before building the complement metric.

In [7]:
luka = df[df["PLAYER_ID"] == LUKA_ID]

if luka.empty:
    print(f"Luka not found in {SEASON} dataset (may have played fewer games than filter threshold)")
else:
    print(f"Luka Doncic — {SEASON} — all merged columns")
    # Transpose for readability
    luka_t = luka.T.reset_index()
    luka_t.columns = ["metric", "value"]
    # Drop null rows for cleaner display
    luka_t = luka_t[luka_t["value"].notna()]
    display(luka_t)

Luka Doncic — 2024-25 — all merged columns


,metric,value
0,PLAYER_ID,1629029
1,PLAYER_NAME,Luka Dončić
2,NICKNAME,Luka
3,TEAM_ID,1610612747
4,TEAM_ABBREVIATION,LAL
...,...,...
143,D_FGM,6.960
144,D_FGA,15.040
145,D_FG_PCT,0.463
146,NORMAL_FG_PCT,0.464


---
## 5. Top 20 by BPM and VORP

Now that BBRef is integrated, we have Box Plus-Minus (BPM) and Value Over Replacement Player (VORP) — the two BBRef metrics most correlated with actual team impact. BPM is a single-season rate; VORP accumulates across minutes played.

Minimum 30 GP filter to exclude sample-size noise.

In [8]:
min_gp = 30

if "BPM" not in df.columns or df["BPM"].isna().all():
    print("BPM not available - BBRef join may have failed. Check scraping logs above.")
else:
    top_bpm = (
        df[df["GP"] >= min_gp]
        .dropna(subset=["BPM"])
        .sort_values("BPM", ascending=False)
        .head(20)[["PLAYER_NAME", "TEAM_ABBREVIATION", "GP", "MIN",
                   "BPM", "OBPM", "DBPM", "VORP",
                   "PER", "WS", "WS_48",
                   "NET_RATING", "USG_PCT"]]
        .reset_index(drop=True)
    )
    top_bpm.index += 1
    print(f"Top 20 by BPM, {SEASON} (min {min_gp} GP)")
    display(top_bpm)

BPM not available - BBRef join may have failed. Check scraping logs above.


In [9]:
if "VORP" not in df.columns or df["VORP"].isna().all():
    print("VORP not available.")
else:
    top_vorp = (
        df[df["GP"] >= min_gp]
        .dropna(subset=["VORP"])
        .sort_values("VORP", ascending=False)
        .head(20)[["PLAYER_NAME", "TEAM_ABBREVIATION", "GP", "MIN",
                   "VORP", "BPM", "WS", "PER",
                   "NET_RATING", "TS_PCT"]]
        .reset_index(drop=True)
    )
    top_vorp.index += 1
    print(f"Top 20 by VORP, {SEASON} (min {min_gp} GP)")
    display(top_vorp)

VORP not available.


---
## 6. Join Failures & Data Quality Audit

Identify players in the nba_api dataset who did NOT get a BBRef row joined (VORP is null). These are likely:
- Name normalization failures (BBRef vs nba_api spelling)
- Two-way contract players not in BBRef's main table
- Players who appeared on the nba_api registry but played 0 games

The manual override dict in `merge.py` handles the ~10 known cases.

In [10]:
# Players with meaningful minutes but no BBRef join
min_gp_check = 20
min_min_check = 15

if "BPM" in df.columns:
    no_bbref = (
        df[
            (df["GP"] >= min_gp_check) &
            (df["MIN"] >= min_min_check) &
            (df["BPM"].isna())
        ]
        .sort_values("MIN", ascending=False)
        [["PLAYER_NAME", "PLAYER_ID", "TEAM_ABBREVIATION", "GP", "MIN"]]
        .reset_index(drop=True)
    )
    print(f"Players with >= {min_gp_check} GP, >= {min_min_check} min/g but no BBRef join: {len(no_bbref)}")
    if not no_bbref.empty:
        print("These likely need name override entries in merge.NAME_OVERRIDES:")
        display(no_bbref)
    else:
        print("No significant join failures found.")
else:
    print("BPM column not present - BBRef not joined. Check scraping logs.")

BPM column not present - BBRef not joined. Check scraping logs.


In [11]:
# Players with salary data who didn't match nba_api (orphaned salary rows)
if "SALARY" in df.columns:
    no_salary = (
        df[
            (df["GP"] >= min_gp_check) &
            (df["MIN"] >= min_min_check) &
            (df["SALARY"].isna())
        ]
        .sort_values("PTS", ascending=False)
        [["PLAYER_NAME", "TEAM_ABBREVIATION", "GP", "MIN", "PTS"]]
        .head(20)
        .reset_index(drop=True)
    )
    print(f"Players missing salary data (min {min_gp_check} GP, {min_min_check} min/g): {len(no_salary)}")
    if not no_salary.empty:
        display(no_salary)
else:
    print("SALARY column not present - salary scraping failed or not yet available.")
    print("Use: from src.data.salaries import generate_salary_template")
    print("     template = generate_salary_template()")
    print("     template.to_csv('data/raw/salaries_2024-25.csv', index=False)")

SALARY column not present - salary scraping failed or not yet available.
Use: from src.data.salaries import generate_salary_template
     template = generate_salary_template()
     template.to_csv('data/raw/salaries_2024-25.csv', index=False)


---
## 7. Lakers Shooting + Defense Breakdown

The key Phase 1 input: catch-and-shoot 3PT efficiency next to defensive FG% suppression for each Laker. This is the first look at roster construction gaps from the complement perspective.

- **CS_CATCH_SHOOT_FG3_PCT** — how efficiently this player shoots off catches (Luka creates)
- **CS_CATCH_SHOOT_FG3A** — volume of CS 3PA (do they actually spot up?)
- **PCT_PLUSMINUS** — opponent FG% vs league average when this player is closest defender (negative = better)

A Luka-complement player wants: high CS_CATCH_SHOOT_FG3_PCT + volume AND negative PCT_PLUSMINUS.

In [12]:
cs_col = "CS_CATCH_SHOOT_FG3_PCT"
cs_vol_col = "CS_CATCH_SHOOT_FG3A"
def_col = "PCT_PLUSMINUS"

lakers_sd_cols = [c for c in [
    "PLAYER_NAME", "GP", "MIN",
    cs_vol_col, cs_col,
    "CS_CATCH_SHOOT_EFG_PCT",
    def_col, "D_FGA",
    "NET_RATING", "USG_PCT",
    # BBRef complement metrics if available
    "WS_48", "BPM",
] if c in lakers.columns]

lakers_sd = lakers[lakers_sd_cols].copy()

print("Lakers 2024-25 — Shooting + Defense Profile")
print(f"Sorted by catch-and-shoot 3PT volume ({cs_vol_col})")
display(
    lakers_sd.sort_values(cs_vol_col, ascending=False).reset_index(drop=True)
    if cs_vol_col in lakers_sd.columns else lakers_sd
)

Lakers 2024-25 — Shooting + Defense Profile
Sorted by catch-and-shoot 3PT volume (CS_CATCH_SHOOT_FG3A)


,PLAYER_NAME,GP,MIN,CS_CATCH_SHOOT_FG3A,CS_CATCH_SHOOT_FG3_PCT,CS_CATCH_SHOOT_EFG_PCT,PCT_PLUSMINUS,D_FGA,NET_RATING,USG_PCT
0,Dorian Finney-Smith,63,28.900,4.800,0.413,0.618,-0.026,11.330,8.800,0.123
1,Rui Hachimura,59,31.700,4.100,0.417,0.623,-0.016,11.530,3.900,0.156
2,Austin Reaves,73,34.900,3.900,0.399,0.596,0.033,15.560,3.800,0.230
3,Dalton Knecht,78,19.200,3.600,0.374,0.562,0.031,7.440,-2.800,0.181
4,Gabe Vincent,72,21.200,3.200,0.372,0.551,-0.009,8.280,1.100,0.131
5,LeBron James,70,34.900,2.900,0.420,0.619,-0.025,11.090,-1.300,0.291
6,Luka Dončić,50,35.400,2.400,0.369,0.565,-0.001,15.040,9.100,0.328
7,Markieff Morris,15,11.000,2.100,0.290,0.426,0.015,4.530,-7.800,0.202
8,Jordan Goodwin,29,18.700,2.000,0.414,0.627,0.054,7.500,-1.200,0.134
9,Quincy Olivari,2,5.200,2.000,0.250,0.375,0.251,1.500,-81.800,0.231


In [13]:
# Annotate: who is a net positive as a Luka complement right now?
# Simple heuristic: CS 3PT% >= 36% AND PCT_PLUSMINUS <= 0
cs_thresh = 0.360
def_thresh = 0.0

if cs_col in lakers.columns and def_col in lakers.columns:
    lakers_annotated = lakers_sd.copy()
    lakers_annotated["shoots_well"] = lakers_annotated.get(cs_col, float("nan")) >= cs_thresh
    lakers_annotated["defends_well"] = lakers_annotated.get(def_col, float("nan")) <= def_thresh
    lakers_annotated["complement_fit"] = (
        lakers_annotated["shoots_well"] & lakers_annotated["defends_well"]
    )

    print(f"CS 3PT% >= {cs_thresh:.0%} AND PCT_PLUSMINUS <= 0: complement-fit Lakers")
    fit = lakers_annotated[lakers_annotated["complement_fit"] == True]
    if fit.empty:
        print("No Laker meets both criteria -- this is the roster gap the model needs to address.")
    else:
        display(fit[["PLAYER_NAME"] + [c for c in [cs_col, cs_vol_col, def_col] if c in fit.columns]])

    print(f"\nShoots well only (CS 3PT% >= {cs_thresh:.0%}):")
    shoots = lakers_annotated[
        lakers_annotated["shoots_well"] == True
    ][["PLAYER_NAME"] + [c for c in [cs_col, cs_vol_col] if c in lakers_annotated.columns]]
    display(shoots) if not shoots.empty else print("  none")

    print(f"\nDefends well only (PCT_PLUSMINUS <= 0):")
    defends = lakers_annotated[
        lakers_annotated["defends_well"] == True
    ][["PLAYER_NAME"] + [c for c in [def_col, "D_FGA"] if c in lakers_annotated.columns]]
    display(defends) if not defends.empty else print("  none")
else:
    print(f"Missing columns: {cs_col} or {def_col} not in dataset.")
    print("Shooting and defense data available:", [c for c in lakers.columns if "CS_" in c or "D_FG" in c or "PCT_PLUS" in c])

CS 3PT% >= 36% AND PCT_PLUSMINUS <= 0: complement-fit Lakers


,PLAYER_NAME,CS_CATCH_SHOOT_FG3_PCT,CS_CATCH_SHOOT_FG3A,PCT_PLUSMINUS
0,Luka Dončić,0.369,2.400,-0.001
2,LeBron James,0.420,2.900,-0.025
3,Rui Hachimura,0.417,4.100,-0.016
4,Dorian Finney-Smith,0.413,4.800,-0.026
5,Gabe Vincent,0.372,3.200,-0.009



Shoots well only (CS 3PT% >= 36%):


,PLAYER_NAME,CS_CATCH_SHOOT_FG3_PCT,CS_CATCH_SHOOT_FG3A
0,Luka Dončić,0.369,2.400
1,Austin Reaves,0.399,3.900
2,LeBron James,0.420,2.900
3,Rui Hachimura,0.417,4.100
4,Dorian Finney-Smith,0.413,4.800
5,Gabe Vincent,0.372,3.200
7,Dalton Knecht,0.374,3.600
9,Jordan Goodwin,0.414,2.000
12,Shake Milton,0.372,1.400



Defends well only (PCT_PLUSMINUS <= 0):


,PLAYER_NAME,PCT_PLUSMINUS,D_FGA
0,Luka Dončić,-0.001,15.040
2,LeBron James,-0.025,11.090
3,Rui Hachimura,-0.016,11.530
4,Dorian Finney-Smith,-0.026,11.330
5,Gabe Vincent,-0.009,8.280
6,Jaxson Hayes,-0.040,9.550
10,Cam Reddish,-0.015,6.500
11,Jarred Vanderbilt,-0.005,6.630
13,Kylor Kelley,-0.003,9.090
15,Trey Jemison III,-0.021,5.560


---
## 8. BBRef Standalone Check

Direct validation that the BBRef scraper works independently of the full merge pipeline. If the cells above show missing BBRef columns, run this to diagnose.

In [17]:
# Direct BBRef fetch for 2024-25 (season ending year = 2025)
bbref_adv = B.get_advanced_stats(2025)
print(f"BBRef advanced rows: {len(bbref_adv)}")
print(f"Columns: {list(bbref_adv.columns)}")
if not bbref_adv.empty:
    display(bbref_adv.head(3))

15:22:23  src.data.cache  INFO  cache miss -- fetching: bbref_advanced_2025
15:22:27  src.data.bbref_stats  WARNING  read_html failed: `Import lxml` failed.  Use pip or conda to install the lxml package.
15:22:27  src.data.cache  INFO  cached 0 rows to bbref_advanced_2025.parquet


BBRef advanced rows: 0
Columns: []


In [15]:
if not bbref_adv.empty and "BPM" in bbref_adv.columns:
    # Sanity check: find Luka in BBRef
    luka_bbref = bbref_adv[bbref_adv["PLAYER_NAME_BBREF"].str.contains("Doncic", case=False, na=False)]
    if not luka_bbref.empty:
        print("Luka in BBRef data:")
        display(luka_bbref)
    else:
        print("Luka not found in BBRef results -- check name column:")
        print(bbref_adv["PLAYER_NAME_BBREF"].head(10).tolist())

---
## 9. EPM Status

EPM from Dunks & Threes is the highest-quality single advanced metric for this project.
If scraping failed (site structure changed or blocked), use the manual loader:

In [16]:
epm = AM.get_epm(SEASON)

if epm.empty:
    print("EPM scraping did not return data for", SEASON)
    print()
    print("Manual download steps:")
    print("  1. Go to https://dunksandthrees.com/epm")
    print("  2. Set season filter to 2024-25")
    print("  3. Use DevTools -> Network to find the data API call, or export CSV")
    print("  4. Save to data/raw/epm_2024-25.csv")
    print("  5. Run: from src.data.advanced_metrics import load_epm")
    print("          epm = load_epm('data/raw/epm_2024-25.csv')")
else:
    print(f"EPM rows: {len(epm)}")
    display(epm.head(10))

15:13:41  src.data.cache  INFO  cache hit: epm_2024-25


EPM scraping did not return data for 2024-25

Manual download steps:
  1. Go to https://dunksandthrees.com/epm
  2. Set season filter to 2024-25
  3. Use DevTools -> Network to find the data API call, or export CSV
  4. Save to data/raw/epm_2024-25.csv
  5. Run: from src.data.advanced_metrics import load_epm
          epm = load_epm('data/raw/epm_2024-25.csv')


---
## 10. Methodology Notes

Key decisions made in this phase:

**Name matching:** Normalized via `merge.normalize_name()` — strips accents, lowercases, removes Jr./III suffixes. A `NAME_OVERRIDES` dict in `merge.py` handles the ~10 players with structurally different spellings across sources (P.J. vs PJ, Nicolas vs Nic). Add overrides there when join failures appear in section 6 above.

**Spine:** nba_api base stats are the spine. Every other source left-joins onto it. Players in BBRef or salary data who don't match an nba_api player are silently dropped — we're building a tool for active NBA players, not a historical archive.

**Traded players:** BBRef keeps a TOT row for players who changed teams mid-season; `bbref_stats.py` keeps only that row, dropping per-team duplicates. nba_api has the same behavior (TEAM_COUNT > 1 players have one row reflecting their season totals).

**System effects caveat:** PCT_PLUSMINUS (defensive FG% suppression) is heavily influenced by teammates, defensive scheme, and opponent quality. A player on a bad defense looks worse here than they actually are. We state this limitation rather than correcting for it — the correction would require lineup-level data not yet in the pipeline.

**Next phase (Phase 1 — Player Valuation):** With PER, WS/48, BPM, VORP, EPM, NET_RATING, TS%, and salary all merged, we have enough signal to build the composite surplus value model. The BBRef metrics provide independent validation of the nba_api metrics and handle edge cases (Jokic's BPM vs PIE ranks differently, which is informative).